# Lab 4.4 &mdash; Bridge Langfuse into a LangChain Agent

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 2 &middot; Module 4 &mdash; Tool Calling &amp; MCP**

### What you'll do
- Turn an MCP tool definition into a LangChain tool &mdash; the adapter is ten lines
- Decide which of 85 published tools your agent is allowed to see
- Watch a naive adapter make the agent loop and then lie about the answer
- Fix it by returning failures as text the model can read

> **How this lab works.** You write real LangChain and MCP code. Fill every `BLANK`, then run
> the **Self-check** cell under each section &mdash; those assert on the *objects you built*
> (a `@tool`, an argument schema, a `ToolMessage`, an `mcp.types.Tool`), so they are
> deterministic and do not depend on the model. Cells marked **Run it for real** put your code
> in front of the sandbox model; that is the part worth watching. The score line is feedback,
> not a grade.

> **The other side of the wire.** Labs 4.1 to 4.3 configured an agent someone else
> wrote. Here you write the agent, and the bridge, yourself.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-4-04")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nSelf-check: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model can reason before it answers, and that reasoning is billed as completion
# tokens. It is off here because tool selection is a short decision and you will make a lot
# of them today. Pass think=True to see the difference for yourself.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

## Concept

`opencode` speaks MCP for you. Your own application does not.

So when the useful tool lives behind an MCP server and your app is LangChain, something has to sit
in between and turn one into the other. That something is smaller than people expect:

| MCP gives you | LangChain wants | how |
|---|---|---|
| `name` | `name` | copy it |
| `description` | `description` | copy it &mdash; this is what the model reads |
| `inputSchema` (JSON Schema) | `args_schema` | **hand it straight over**, no Pydantic needed |
| `tools/call` over HTTP | a Python callable | one function that posts and returns text |

`langchain-mcp-adapters` exists and would do this for you. It is **not installed here**, on purpose:
the packaged adapter hides exactly the decision this lab is about, which is what your tool returns
when the call goes wrong.

## Section 1 &mdash; One MCP tool becomes one LangChain tool

The Langfuse server publishes its tools as plain dictionaries. Below is a real one, captured from
`tools/list`, so this section needs no network.

Fill in the blank: **which field carries the sentence the model reads when deciding to call this
tool at all?**

In [ ]:
from langchain_core.tools import StructuredTool

# A real spec, captured from the Langfuse MCP server's tools/list.
SPEC = {
    "name": "listPrompts",
    "description": "List prompts in the current Langfuse project with cursor-based pagination.",
    "inputSchema": {
        "type": "object",
        "properties": {"limit": {"type": "integer", "description": "How many to return"},
                       "page":  {"type": "integer", "description": "1-based page number"}},
        "required": [],
    },
}


def to_langchain(spec: dict, call_fn) -> StructuredTool:
    """One MCP tool definition -> one LangChain tool.

    `call_fn(**kwargs)` is whatever actually performs tools/call. Keeping it a
    parameter is what lets the self-checks below run with no network at all.
    """
    return StructuredTool.from_function(
        func=call_fn,
        name=spec["name"],
        # Three fields cross to the model. Two of them are structural. Which one
        # is the prose that decides whether this tool gets chosen?
        description=spec[BLANK],
        # MCP publishes JSON Schema and StructuredTool accepts JSON Schema.
        args_schema=spec.get("inputSchema") or {"type": "object", "properties": {}},
    )

In [ ]:
# ---- Self-check: the object you built, no model and no network ----
def _fake_call(**kwargs):
    return f"called with {kwargs}"

check("to_langchain returns a StructuredTool",
      lambda: isinstance(to_langchain(SPEC, _fake_call), StructuredTool))
check("the name comes across unchanged",
      lambda: to_langchain(SPEC, _fake_call).name == "listPrompts")
check("the description is the prose, not the name or the schema",
      lambda: to_langchain(SPEC, _fake_call).description.startswith("List prompts"),
      "that sentence is the only thing that tells the model WHEN to use this tool")
check("MCP's JSON Schema became the tool's arguments",
      lambda: set(to_langchain(SPEC, _fake_call).args) == {"limit", "page"})

from langchain_core.utils.function_calling import convert_to_openai_tool
check("and it renders onto the wire like any other LangChain tool",
      lambda: convert_to_openai_tool(to_langchain(SPEC, _fake_call))["function"]["name"] == "listPrompts")
score()

## Section 2 &mdash; What your tool returns when the call fails

Here is a run of this exact agent, built with an adapter that returned `""` whenever the server
rejected a call. The question was *&ldquo;how many observations are in this project?&rdquo;*

```
  queryMetrics {"metrics":[{"measure":"id","aggregation":"count"}]}   -> ""
  queryMetrics {"metrics":[{"measure":"id","aggregation":"count"}]}   -> ""
  queryMetrics {"metrics":[{"measure":"id","aggregation":"count"}]}   -> ""
  queryMetrics {"metrics":[{"measure":"id","aggregation":"count"}]}   -> ""
  queryMetrics {"metrics":[{"measure":"id","aggregation":"count"}]}   -> ""

  "there are 0 observations in this project"
```

There are 32. `measure: "id"` is not valid, and the server said so &mdash; but the adapter threw the
message away and handed back an empty string. With nothing to correct from, the model repeated
itself and then reported the emptiness as an answer.

**A tool that fails silently does not produce a failure. It produces a confident wrong answer.**

Fill in what a failed call should hand back.

In [ ]:
def mcp_result_to_text(envelope: dict) -> str:
    """Turn a raw JSON-RPC envelope from tools/call into what the model will read.

    Three cases, and only one of them is success:
      * a protocol-level "error"      -- the request was malformed or rejected
      * a result with isError: true   -- the tool ran and failed
      * a result with content         -- the tool ran and worked
    """
    if "error" in envelope:
        message = str(envelope["error"].get("message", envelope["error"]))
        return BLANK                          # what should the model see here?

    result = envelope.get("result", {})
    text = "".join(part.get("text", "") for part in result.get("content", []))

    if result.get("isError"):
        return "ERROR: " + text[:600]
    return text[:2000] or "(the call succeeded and returned nothing)"

In [ ]:
# ---- Self-check: real envelopes, captured from the server. Still no network ----
REJECTED = {"jsonrpc": "2.0", "id": 1, "error": {"code": -32600,
            "message": "Invalid metric id. Must be one of count,traceId,latency,totalTokens"}}
FAILED   = {"jsonrpc": "2.0", "id": 1, "result": {
            "content": [{"type": "text", "text": "project not found"}], "isError": True}}
WORKED   = {"jsonrpc": "2.0", "id": 1, "result": {
            "content": [{"type": "text", "text": '{"data":[{"count_count":32}]}'}]}}

check("a rejected call comes back as readable text, not empty",
      lambda: mcp_result_to_text(REJECTED).strip() != "",
      "an empty string is what made the agent loop and then invent an answer")
check("and it carries the reason the model needs to fix itself",
      lambda: "Invalid metric" in mcp_result_to_text(REJECTED),
      "the server listed the valid metrics -- pass that on and the model can retry correctly")
check("a failed tool is distinguishable from a successful one",
      lambda: mcp_result_to_text(FAILED) != mcp_result_to_text(WORKED))
check("a successful call returns its content",
      lambda: "count_count" in mcp_result_to_text(WORKED))
check("success is never reported as an error",
      lambda: not mcp_result_to_text(WORKED).startswith("ERROR"))
score()

## Run it for real

Now the two halves meet: a live session against Langfuse, your adapter, and an agent that has never
heard of MCP.

**The server publishes 85 tools.** You are going to bind three. That is not a limitation, it is the
decision &mdash; your agent's capabilities are whatever you hand it, and every tool you add is
context on every turn plus one more thing it might choose wrongly.

In [ ]:
import base64, urllib.request

LF_HOST = os.environ.get("LANGFUSE_HOST", "").rstrip("/")
LF_PK   = os.environ.get("LANGFUSE_PUBLIC_KEY", "")
LF_SK   = os.environ.get("LANGFUSE_SECRET_KEY", "")
LF_URL  = LF_HOST + "/api/public/mcp" if LF_HOST else ""
LF_AUTH = base64.b64encode(f"{LF_PK}:{LF_SK}".encode()).decode() if (LF_PK and LF_SK) else ""
_session = {"id": None}


def langfuse_ready() -> bool:
    return bool(LF_HOST and LF_PK and LF_SK)


def rpc(method: str, params: dict = None, timeout: int = 120) -> dict:
    """One JSON-RPC call. Returns the WHOLE envelope so failures survive."""
    body = json.dumps({"jsonrpc": "2.0", "id": 1, "method": method, "params": params or {}}).encode()
    req = urllib.request.Request(LF_URL, data=body, method="POST")
    req.add_header("Content-Type", "application/json")
    req.add_header("Accept", "application/json, text/event-stream")
    req.add_header("Authorization", "Basic " + LF_AUTH)
    if _session["id"]:
        req.add_header("Mcp-Session-Id", _session["id"])
    with urllib.request.urlopen(req, timeout=timeout) as r:
        raw, sid = r.read().decode(), r.headers.get("mcp-session-id")
    if sid:
        _session["id"] = sid
    for line in raw.splitlines():
        if line.startswith("data:"):
            raw = line[5:].strip()
            break
    return json.loads(raw)


# Three of eighty-five. Enough to answer a real question, small enough to reason about.
WANTED = ["getMetricsSchema", "queryMetrics", "listPrompts"]


def build_tools():
    rpc("initialize", {"protocolVersion": "2025-06-18", "capabilities": {},
                       "clientInfo": {"name": "lab-4-4", "version": "1.0"}})
    published = rpc("tools/list", {})["result"]["tools"]

    def bind(spec):
        def call(**kwargs):
            return mcp_result_to_text(rpc("tools/call", {"name": spec["name"], "arguments": kwargs}))
        return to_langchain(dict(spec, description=spec.get("description", "")[:900]), call)

    return published, [bind(s) for s in published if s["name"] in WANTED]


if langfuse_ready():
    published, tools = guard(build_tools, (None, None))
    if tools:
        print(f"the server publishes {len(published)} tools")
        print(f"you bound          {len(tools)}: {[t.name for t in tools]}")
        print(f"\nschema you are NOT sending on every turn: "
              f"{(len(json.dumps(published)) - len(json.dumps([t.args for t in tools]))) // 4:,} tokens")
else:
    print("Langfuse is not configured in this sandbox - set LANGFUSE_HOST / _PUBLIC_KEY / _SECRET_KEY")

### The agent

Nothing below mentions MCP. `create_agent` is handed three LangChain tools and has no idea where
they came from &mdash; which is the entire point of having written the adapter.

In [ ]:
def ask_langfuse(question: str):
    from langchain.agents import create_agent
    agent = create_agent(
        model=get_llm(),
        tools=tools,
        system_prompt=("You answer questions about a Langfuse project using the tools provided. "
                       "Call getMetricsSchema before guessing field names. "
                       "If a tool returns an ERROR, read it and correct your arguments. Be brief."),
    )
    out = agent.invoke({"messages": [("user", question)]})
    for m in out["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("  call:", tc["name"], json.dumps(tc["args"])[:96])
        if type(m).__name__ == "ToolMessage" and str(m.content).startswith("ERROR"):
            print("   ->", str(m.content)[:110])
    return out["messages"][-1].content


if llm_ready() and langfuse_ready() and tools:
    print(guard(lambda: ask_langfuse(
        "How many observations are in this project? Answer with the number.")))
else:
    print("Run it for real needs both the model and Langfuse configured. See the setup cell.")

### What to look for

Watch the call trace, not just the answer.

If the model gets an argument wrong &mdash; and it very often does on `queryMetrics`, whose valid
measures it cannot guess &mdash; your adapter hands back the server's rejection *including the list
of valid values*, and the next call is usually right. That recovery is not the model being clever.
**It is your ten-line adapter choosing to pass the error on.**

Swap `mcp_result_to_text` for one that returns `""` on failure and run it again if you want to see
the difference. You will get five identical calls and a confident wrong number.

## Your turn

- Add `listObservations` to `WANTED` and ask something that needs two tools. Watch it chain.
- Truncate the description to 40 characters in `to_langchain` and re-run. Same tools, same model,
  worse selection &mdash; the prose was doing more work than the schema.
- Bind all 85 and see what it costs you in tokens before the question is even read.
- Point the same adapter at the Jira server from Lab 4.1. Nothing about it is Langfuse-specific,
  which is what a protocol is for.